In [10]:
import umap
import pickle
import hdbscan
import numpy as np
import json, os, textwrap
import matplotlib.pyplot as plt
from collections import defaultdict, Counter


In [11]:
embeddings_file = "dataset/embeddings.pkl" 
sentences_file = "dataset/sentences.txt"
output = "dataset/" 

In [12]:
sentences = []
with open(sentences_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            sentences.append(line)
 

In [13]:
# load embeddings
with open(embeddings_file, 'rb') as f:
    embeddings = pickle.load(f)
 
# convert to numpy array just in case
embeddings = np.array(embeddings)

In [14]:
reducer = umap.UMAP(n_components=12, metric='cosine', random_state=42)
embeddings_2d = reducer.fit_transform(embeddings)


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [15]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=6, metric='euclidean')
labels = clusterer.fit_predict(embeddings_2d)

In [16]:
num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
num_noise = list(labels).count(-1)

print(f"total points {len(sentences)}")
print(f"found {num_clusters} clusters")
print(f"noise points (unclustered): {num_noise}")

total points 3408
found 122 clusters
noise points (unclustered): 761


# Convert ouput to json

In [17]:
def save_results(labels, sentences):
    cluster_map = defaultdict(list)
    for i, label in enumerate(labels):
        cluster_map[int(label)].append({"id": i, "sentence": sentences[i]})

    out_path = os.path.join(output, f"clusters.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(cluster_map, f, ensure_ascii=False, indent=2)
    print(f"Results saved -> {out_path}")

    np.save(os.path.join(output, f"labels.npy"), labels)

In [18]:
save_results(labels, sentences)

Results saved -> dataset/clusters.json


# parameter tuning

In [19]:
import optuna
import umap
import hdbscan
import numpy as np
from hdbscan.validity import validity_index

In [21]:
from hdbscan.validity import validity_index

n_samples, n_dims = embeddings.shape

def objective(trial):
    n_components     = trial.suggest_int("n_components",     2, min(n_dims, 50))
    min_cluster_size = trial.suggest_int("min_cluster_size", 2, max(5, n_samples // 20))

    reducer = umap.UMAP(n_components=n_components, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(embeddings)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', gen_min_span_tree=True)
    labels = clusterer.fit_predict(embeddings_2d)

    n_noise = int(np.sum(labels == -1))
    trial.set_user_attr("n_noise", n_noise)

    if len(set(labels) - {-1}) < 2:
        return -1.0

    dbcv_score = validity_index(embeddings_2d.astype(np.float64), labels)
    print(f"Trial {trial.number} | n_components={n_components} min_cluster_size={min_cluster_size} | noise={n_noise} dbcv={dbcv_score:.4f}")

    return dbcv_score

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50)

# print(study.best_params)
# print(study.best_value)
# print("n_noise:", study.best_trial.user_attrs["n_noise"])